# Phase 1 — Data Setup & Exploratory Data Analysis
## Netflix Prize Recommendation System

**Goal of this notebook:**
- Parse the raw `combined_data_*.txt` files into a clean, flat DataFrame
- Join movie metadata from `movie_titles.csv`
- Explore the dataset deeply: rating distributions, user behaviour, movie popularity, sparsity, and time patterns
- Set up the `probe.txt` test split for later evaluation

Work through each cell in order. Every section has a comment explaining *why* we do it this way.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})

print("All imports OK")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

All imports OK
pandas 3.0.3 | numpy 2.4.6


## 1. File Paths

Set these to match wherever your Kaggle download lives.
If you are on Kaggle itself, the paths are already `/kaggle/input/netflix-prize-data/`.

In [2]:
# ── CONFIGURE THIS ────────────────────────────────────────────────────────────
# Change DATA_DIR to the folder containing the Netflix Prize files
DATA_DIR = "/kaggle/input/netflix-prize-data"   # Kaggle default
# DATA_DIR = "./data"                           # Local alternative

COMBINED_FILES = [
    os.path.join(DATA_DIR, f"combined_data_{i}.txt") for i in range(1, 5)
]
MOVIE_TITLES_FILE = os.path.join(DATA_DIR, "movie_titles.csv")
PROBE_FILE        = os.path.join(DATA_DIR, "probe.txt")

# Check files exist
for f in COMBINED_FILES + [MOVIE_TITLES_FILE, PROBE_FILE]:
    status = "OK" if os.path.exists(f) else "MISSING"
    size_mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f"  [{status}]  {os.path.basename(f):30s}  {size_mb:7.1f} MB")

  [MISSING]  combined_data_1.txt                 0.0 MB
  [MISSING]  combined_data_2.txt                 0.0 MB
  [MISSING]  combined_data_3.txt                 0.0 MB
  [MISSING]  combined_data_4.txt                 0.0 MB
  [MISSING]  movie_titles.csv                    0.0 MB
  [MISSING]  probe.txt                           0.0 MB


## 2. Parsing `combined_data_*.txt` → Flat DataFrame

### Why is this non-trivial?

The raw file format is **not** a CSV. It looks like:

```
1:
1488844,3,2005-09-06
822109,5,2005-05-13
885013,4,2005-10-19
2:
1530469,4,2003-11-05
...
```

Every line ending in `:` is a **movie ID header**. The lines below it — until the next header — are ratings for that movie in the format `user_id, rating, date`.

### Strategy: line-by-line streaming parser

We **cannot** just call `pd.read_csv()` on this. We need to:
1. Read line by line
2. Track the current `movie_id` whenever we see a header line
3. For all other lines, append `(movie_id, user_id, rating, date)`

We collect into Python lists (much faster than appending to a DataFrame row-by-row),
then build the DataFrame once at the end.

### Practical tip on file size
The full dataset (all 4 files) is ~3.5 GB and has 100M+ rows.
For development and EDA, **start with just `combined_data_1.txt`** (~500 MB, ~24M rows).
Once your pipeline is solid, load all four.

The parameter `max_rows` below lets you limit rows during testing.

In [ ]:
def parse_combined_data(filepaths, max_rows=None):
    """
    Parse one or more combined_data_*.txt files into a flat DataFrame.

    Parameters
    ----------
    filepaths : list[str]
        Paths to combined_data files to parse.
    max_rows : int or None
        If set, stop after this many rating rows (useful for quick tests).

    Returns
    -------
    pd.DataFrame with columns: movie_id, user_id, rating, date
    """
    movie_ids = []
    user_ids  = []
    ratings   = []
    dates     = []

    current_movie_id = None
    total_rows = 0
    t0 = time.time()

    for filepath in filepaths:
        fname = os.path.basename(filepath)
        print(f"  Parsing {fname} ...", end="", flush=True)
        file_rows = 0

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                # ── Movie header line, e.g. "1:" ──────────────────────────────
                if line.endswith(":"):
                    current_movie_id = int(line[:-1])
                    continue

                # ── Rating line, e.g. "1488844,3,2005-09-06" ─────────────────
                parts = line.split(",")
                movie_ids.append(current_movie_id)
                user_ids.append(int(parts[0]))
                ratings.append(int(parts[1]))
                dates.append(parts[2])

                file_rows  += 1
                total_rows += 1

                if max_rows and total_rows >= max_rows:
                    print(f" {file_rows:,} rows (limit reached)  [{time.time()-t0:.1f}s]")
                    # Build DataFrame and return early
                    df = pd.DataFrame({
                        "movie_id": pd.array(movie_ids, dtype="int32"),
                        "user_id":  pd.array(user_ids,  dtype="int32"),
                        "rating":   pd.array(ratings,   dtype="int8"),
                        "date":     pd.to_datetime(dates),
                    })
                    return df

        print(f" {file_rows:,} rows  [{time.time()-t0:.1f}s]")

    print(f"\nTotal rows parsed: {total_rows:,}  |  Elapsed: {time.time()-t0:.1f}s")

    # ── Build DataFrame with memory-efficient dtypes ──────────────────────────
    # int32 instead of int64 halves the memory for IDs
    # int8  is enough for ratings 1-5
    df = pd.DataFrame({
        "movie_id": pd.array(movie_ids, dtype="int32"),
        "user_id":  pd.array(user_ids,  dtype="int32"),
        "rating":   pd.array(ratings,   dtype="int8"),
        "date":     pd.to_datetime(dates),
    })
    return df


# ── Parse (start with file 1 only for EDA) ────────────────────────────────────
# For full dataset, replace with COMBINED_FILES (all 4 files, ~25 min on Kaggle)
print("Parsing combined_data_1.txt ...")
df_ratings = parse_combined_data([COMBINED_FILES[0]])

print(f"\nShape: {df_ratings.shape}")
print(f"Memory usage: {df_ratings.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_ratings.head(10)

## 3. Load Movie Metadata

`movie_titles.csv` maps each `movie_id` to a title and release year.

**Known issue:** The file has encoding problems — some movie titles contain non-ASCII
characters that break `utf-8`. We use `latin-1` (ISO-8859-1) which handles all of them.

In [ ]:
# The file has no header, so we supply column names manually.
# Encoding: latin-1 handles special characters in titles.
df_movies = pd.read_csv(
    MOVIE_TITLES_FILE,
    header=None,
    names=["movie_id", "year", "title"],
    encoding="latin-1",
    on_bad_lines="skip",   # a handful of lines have extra commas in titles
)

# Coerce year to numeric (some entries are NULL)
df_movies["year"] = pd.to_numeric(df_movies["year"], errors="coerce")
df_movies["movie_id"] = df_movies["movie_id"].astype("int32")

print(f"Movies loaded: {len(df_movies):,}")
print(f"Year range:    {df_movies['year'].min():.0f} – {df_movies['year'].max():.0f}")
print(f"Missing years: {df_movies['year'].isna().sum()}")
df_movies.head(8)

## 4. Join Ratings ← Movie Metadata

A simple left join on `movie_id` gives us the title and year on every rating row.
We use a left join (not inner) to keep all ratings even if a movie_id is somehow
absent from the titles file.

In [ ]:
df = df_ratings.merge(df_movies, on="movie_id", how="left")

# Quick sanity check
missing_title = df["title"].isna().sum()
print(f"Ratings without a matching title: {missing_title:,}")

# Extract year and month from date for time-series analysis later
df["year_rated"] = df["date"].dt.year.astype("int16")
df["month_rated"] = df["date"].dt.month.astype("int8")

print(f"\nFinal DataFrame shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(5)

## 5. Basic Dataset Statistics

Before plotting anything, let's get the hard numbers.
These go directly into your EDA section of the report.

In [ ]:
n_ratings = len(df)
n_users   = df["user_id"].nunique()
n_movies  = df["movie_id"].nunique()
n_cells   = n_users * n_movies
density   = n_ratings / n_cells * 100

avg_rating = df["rating"].mean()
median_rating = df["rating"].median()

print("=" * 50)
print("  DATASET SUMMARY (combined_data_1.txt)")
print("=" * 50)
print(f"  Total ratings   : {n_ratings:>15,.0f}")
print(f"  Unique users    : {n_users:>15,.0f}")
print(f"  Unique movies   : {n_movies:>15,.0f}")
print(f"  Possible cells  : {n_cells:>15,.0f}")
print(f"  Matrix density  : {density:>14.4f}%")
print(f"  Sparsity        : {100-density:>14.4f}%")
print(f"  Avg rating      : {avg_rating:>15.3f}")
print(f"  Median rating   : {median_rating:>15.1f}")
print(f"  Date range      : {df['date'].min().date()} → {df['date'].max().date()}")
print("=" * 50)

## 6. Rating Distribution

**What to look for:**
- Is there a positive bias? (people tend to rate things they *chose* to watch, so we often see more 4s and 5s)
- Are there very few 1s? (negative selection bias — people rarely finish something they hate)
- This distribution affects model training: a model that always predicts 4 will have deceptively low error

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: count bar chart
rating_counts = df["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values,
            color=sns.color_palette("Blues_d", 5), edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Rating (1–5 stars)")
axes[0].set_ylabel("Number of ratings")
axes[0].set_title("Rating distribution — count")
for i, (star, count) in enumerate(rating_counts.items()):
    axes[0].text(star, count + n_ratings * 0.002,
                 f"{count/n_ratings*100:.1f}%", ha="center", fontsize=9)

# Right: cumulative distribution
cumulative = rating_counts.cumsum() / rating_counts.sum()
axes[1].plot(cumulative.index, cumulative.values * 100,
             marker="o", color="#2a6fad", linewidth=2)
axes[1].axhline(50, color="grey", linestyle="--", linewidth=0.8, alpha=0.7)
axes[1].set_xlabel("Rating (1–5 stars)")
axes[1].set_ylabel("Cumulative %")
axes[1].set_title("Cumulative distribution")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle("Rating distribution", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("plot_01_rating_distribution.png", bbox_inches="tight")
plt.show()

print("\nRating breakdown:")
for star, count in rating_counts.items():
    print(f"  {star} star(s): {count:>10,.0f}  ({count/n_ratings*100:.2f}%)")

## 7. User Activity Distribution

**Why this matters:**
- Users who rated only 1–2 movies are nearly impossible to build a profile for → cold-start problem
- Power users (who rated 1000+) dominate the training signal and can skew models
- This distribution directly affects how you'll sample your train/test split

In [ ]:
ratings_per_user = df.groupby("user_id")["rating"].count()

print("Ratings per user — summary statistics:")
print(ratings_per_user.describe(percentiles=[0.01, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram (log scale on x)
axes[0].hist(ratings_per_user, bins=100, color="#4a7fb5", edgecolor="none", alpha=0.85)
axes[0].set_xscale("log")
axes[0].set_xlabel("Ratings per user (log scale)")
axes[0].set_ylabel("Number of users")
axes[0].set_title("User activity distribution")
axes[0].axvline(ratings_per_user.median(), color="red", linestyle="--",
                linewidth=1, label=f"Median = {ratings_per_user.median():.0f}")
axes[0].legend(fontsize=9)

# Right: CDF
sorted_counts = np.sort(ratings_per_user.values)
cdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
axes[1].plot(sorted_counts, cdf * 100, color="#4a7fb5", linewidth=1.5)
axes[1].set_xscale("log")
axes[1].set_xlabel("Ratings per user (log scale)")
axes[1].set_ylabel("Cumulative % of users")
axes[1].set_title("User activity CDF")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

# Annotate key thresholds
for pct in [25, 50, 75, 90]:
    val = np.percentile(ratings_per_user, pct)
    axes[1].axvline(val, color="grey", linestyle=":", linewidth=0.8, alpha=0.7)
    axes[1].text(val * 1.05, pct - 3, f"p{pct}={val:.0f}", fontsize=8, color="grey")

plt.tight_layout()
plt.savefig("plot_02_user_activity.png", bbox_inches="tight")
plt.show()

# Segment users by activity level
cold_users   = (ratings_per_user < 5).sum()
normal_users = ((ratings_per_user >= 5) & (ratings_per_user < 200)).sum()
power_users  = (ratings_per_user >= 200).sum()
print(f"\nCold-start users  (<5 ratings):   {cold_users:>8,.0f}  ({cold_users/n_users*100:.1f}%)")
print(f"Normal users   (5–199 ratings):   {normal_users:>8,.0f}  ({normal_users/n_users*100:.1f}%)")
print(f"Power users     (200+ ratings):   {power_users:>8,.0f}  ({power_users/n_users*100:.1f}%)")

## 8. Movie Popularity Distribution

**Why this matters:**
- Popular movies (thousands of ratings) get great model fits — we have dense signal
- Niche movies (few ratings) are hard to recommend reliably
- This is the **long-tail problem**: a few movies dominate rating counts

In [ ]:
ratings_per_movie = df.groupby("movie_id")["rating"].count().sort_values(ascending=False)
avg_rating_per_movie = df.groupby("movie_id")["rating"].mean()

print("Ratings per movie — summary statistics:")
print(ratings_per_movie.describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: log-log histogram (long tail characteristic)
axes[0].hist(ratings_per_movie, bins=100, color="#5b9e5b", edgecolor="none", alpha=0.85)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Ratings per movie (log scale)")
axes[0].set_ylabel("Number of movies (log scale)")
axes[0].set_title("Movie popularity (log-log)")
axes[0].axvline(ratings_per_movie.median(), color="red", linestyle="--",
                linewidth=1, label=f"Median = {ratings_per_movie.median():.0f}")
axes[0].legend(fontsize=9)

# Right: top-20 most rated movies
top20 = ratings_per_movie.head(20).reset_index()
top20 = top20.merge(df_movies[["movie_id","title"]], on="movie_id", how="left")
top20["short_title"] = top20["title"].str[:28]
axes[1].barh(range(20), top20["rating"], color="#5b9e5b", alpha=0.85)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20["short_title"], fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel("Number of ratings")
axes[1].set_title("Top 20 most-rated movies")

plt.tight_layout()
plt.savefig("plot_03_movie_popularity.png", bbox_inches="tight")
plt.show()

# Concentration check: what % of ratings come from top 100 movies?
top100_share = ratings_per_movie.head(100).sum() / n_ratings * 100
top1_share   = ratings_per_movie.head(1).sum()  / n_ratings * 100
print(f"\nTop 1 movie accounts for  {top1_share:.2f}% of all ratings")
print(f"Top 100 movies account for {top100_share:.2f}% of all ratings")

## 9. Rating Quality vs Popularity

Do popular movies actually get better ratings, or is it just volume?
This scatter plot reveals whether popularity and quality are correlated.

**Insight to note:** We often see a "quality floor" — very popular movies rarely get
below 3 stars (audiences self-select), but very niche movies have wildly variable ratings.

In [ ]:
movie_stats = pd.DataFrame({
    "n_ratings": ratings_per_movie,
    "avg_rating": avg_rating_per_movie,
}).reset_index()
movie_stats = movie_stats.merge(df_movies[["movie_id","title","year"]], on="movie_id", how="left")

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(
    movie_stats["n_ratings"],
    movie_stats["avg_rating"],
    alpha=0.2, s=6, c=movie_stats["avg_rating"],
    cmap="RdYlGn", vmin=1, vmax=5
)
ax.set_xscale("log")
ax.set_xlabel("Number of ratings (log scale)")
ax.set_ylabel("Average rating")
ax.set_title("Movie quality vs popularity")
ax.axhline(df["rating"].mean(), color="grey", linestyle="--",
           linewidth=0.8, label=f"Overall mean = {df['rating'].mean():.2f}")
ax.legend(fontsize=9)
plt.colorbar(sc, ax=ax, label="Average rating", shrink=0.7)

plt.tight_layout()
plt.savefig("plot_04_quality_vs_popularity.png", bbox_inches="tight")
plt.show()

# Pearson correlation
corr, pval = stats.pearsonr(
    np.log1p(movie_stats["n_ratings"]),
    movie_stats["avg_rating"]
)
print(f"Pearson correlation (log popularity vs avg rating): r = {corr:.3f}, p = {pval:.3e}")

## 10. Ratings Over Time

Netflix data spans **October 1998 to December 2005**.
Temporal patterns matter for recommendation systems because:
- User preferences drift over time
- When splitting train/test, you should ideally **split by time** (train on earlier, test on later)
  rather than a random split — otherwise you leak future information

In [ ]:
monthly_counts = (df.groupby(["year_rated","month_rated"])
                    .size()
                    .reset_index(name="count"))
monthly_counts["period"] = pd.to_datetime(
    monthly_counts["year_rated"].astype(str) + "-" +
    monthly_counts["month_rated"].astype(str).str.zfill(2)
)
monthly_counts = monthly_counts.sort_values("period")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

# Top: volume over time
axes[0].fill_between(monthly_counts["period"],
                     monthly_counts["count"] / 1e3,
                     alpha=0.6, color="#4a7fb5")
axes[0].plot(monthly_counts["period"],
             monthly_counts["count"] / 1e3,
             color="#2a5f9a", linewidth=1)
axes[0].set_ylabel("Ratings (thousands)")
axes[0].set_title("Rating volume over time")

# Bottom: avg rating per month
monthly_avg = (df.groupby(["year_rated","month_rated"])["rating"]
                 .mean()
                 .reset_index(name="avg_rating"))
monthly_avg["period"] = pd.to_datetime(
    monthly_avg["year_rated"].astype(str) + "-" +
    monthly_avg["month_rated"].astype(str).str.zfill(2)
)
monthly_avg = monthly_avg.sort_values("period")
axes[1].plot(monthly_avg["period"], monthly_avg["avg_rating"],
             color="#5b9e5b", linewidth=1.5)
axes[1].axhline(df["rating"].mean(), color="grey", linestyle="--",
                linewidth=0.8, alpha=0.7)
axes[1].set_ylabel("Average rating")
axes[1].set_title("Average rating over time")
axes[1].set_ylim(3.0, 4.5)

plt.tight_layout()
plt.savefig("plot_05_ratings_over_time.png", bbox_inches="tight")
plt.show()

# Key insight: identify the spike year
peak = monthly_counts.loc[monthly_counts["count"].idxmax()]
print(f"Peak rating month: {peak['period'].strftime('%Y-%m')}  ({peak['count']:,.0f} ratings)")

## 11. Movie Release Year Analysis

Older movies and newer movies behave very differently in terms of how many ratings
they accumulate and what average rating they receive.

In [ ]:
# Join movie year onto ratings
df_year = df.dropna(subset=["year"])
year_stats = (df_year.groupby("year")
              .agg(n_ratings=("rating","count"), avg_rating=("rating","mean"))
              .reset_index())
year_stats = year_stats[year_stats["year"] >= 1980]  # sparse before 1980

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(year_stats["year"], year_stats["n_ratings"] / 1e3,
            color="#9b59b6", alpha=0.8, edgecolor="none")
axes[0].set_xlabel("Movie release year")
axes[0].set_ylabel("Ratings (thousands)")
axes[0].set_title("Ratings by movie release year")

axes[1].scatter(year_stats["year"], year_stats["avg_rating"],
                c=year_stats["avg_rating"], cmap="RdYlGn",
                vmin=2.5, vmax=4.5, s=40, alpha=0.9)
axes[1].axhline(df["rating"].mean(), color="grey", linestyle="--",
                linewidth=0.8, alpha=0.7)
axes[1].set_xlabel("Movie release year")
axes[1].set_ylabel("Average rating")
axes[1].set_title("Average rating by release year")

plt.tight_layout()
plt.savefig("plot_06_release_year.png", bbox_inches="tight")
plt.show()

## 12. Visualising Matrix Sparsity

A recommendation system works on a **users × movies matrix**.
98.8% of that matrix is empty (no rating). This section makes that tangible
by sampling a small slice and visualising it as a heatmap.

This is also a great visual for your report — it shows *why* matrix factorization
is needed (you can't just look up answers — most of the matrix is unknown).

In [ ]:
# Sample a small slice: top-50 users by activity × top-50 movies by popularity
top_users  = ratings_per_user.nlargest(50).index.tolist()
top_movies = ratings_per_movie.head(50).index.tolist()

slice_df = df[(df["user_id"].isin(top_users)) & (df["movie_id"].isin(top_movies))]

pivot = slice_df.pivot_table(
    index="user_id", columns="movie_id", values="rating", aggfunc="mean"
)
# Reindex to enforce ordering
pivot = pivot.reindex(index=top_users, columns=top_movies)

# Build a mask for cells with no rating
mask = pivot.isna()

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot.fillna(0),
    mask=False,
    cmap="RdYlGn",
    vmin=1, vmax=5,
    linewidths=0.0,
    ax=ax,
    cbar_kws={"label": "Rating (0 = missing)"},
    xticklabels=False,
    yticklabels=False,
)
# Overlay missing cells in a neutral grey
missing_overlay = pivot.copy()
missing_overlay[:] = np.nan
missing_overlay[mask] = 0
sns.heatmap(
    mask.astype(float),
    cmap=sns.color_palette(["#e0e0e0"], as_cmap=True),
    alpha=0.6,
    ax=ax,
    cbar=False,
    xticklabels=False,
    yticklabels=False,
)

actual_density = (~mask).sum().sum() / (50*50) * 100
ax.set_title(
    f"User–Movie rating matrix (top-50 × top-50 slice)\n"
    f"Filled: {actual_density:.1f}%  |  Missing: {100-actual_density:.1f}% (grey)"
)
ax.set_xlabel("Movies →")
ax.set_ylabel("Users →")

plt.tight_layout()
plt.savefig("plot_07_sparsity_heatmap.png", bbox_inches="tight")
plt.show()

print(f"\nFull matrix: {n_users:,} users × {n_movies:,} movies")
print(f"Possible entries: {n_users * n_movies:,.0f}")
print(f"Observed ratings: {n_ratings:,.0f}")
print(f"Density: {n_ratings / (n_users * n_movies) * 100:.4f}%")

## 13. Setting Up the Probe Test Split

`probe.txt` contains user-movie pairs whose ratings *are* in the training data.
Netflix provided this to let competitors evaluate models before the final submission.

**We use it as our test set:**
1. Parse probe.txt to get all (movie_id, user_id) pairs
2. Look up their actual ratings from `df`
3. Use the remaining rows as training data

This is the **correct evaluation strategy** — never split randomly on this dataset
without accounting for the probe set.

In [ ]:
def parse_probe(filepath):
    """
    Parse probe.txt into a DataFrame of (movie_id, user_id) pairs.
    Format mirrors combined_data: movie headers followed by user_id lines.
    """
    movie_ids = []
    user_ids  = []
    current_movie_id = None

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.endswith(":"):
                current_movie_id = int(line[:-1])
            else:
                user_ids.append(int(line))
                movie_ids.append(current_movie_id)

    return pd.DataFrame({
        "movie_id": pd.array(movie_ids, dtype="int32"),
        "user_id":  pd.array(user_ids,  dtype="int32"),
    })


probe_pairs = parse_probe(PROBE_FILE)
print(f"Probe pairs loaded: {len(probe_pairs):,}")

# ── Merge probe pairs with actual ratings ────────────────────────────────────
# We need to match on (movie_id, user_id, date) because the same user can
# rate the same movie multiple times. In practice, duplicates are rare, so
# we take the first match.
probe_with_ratings = probe_pairs.merge(
    df[["movie_id", "user_id", "rating", "date"]],
    on=["movie_id", "user_id"],
    how="left",
).drop_duplicates(subset=["movie_id", "user_id"])

missing_probe = probe_with_ratings["rating"].isna().sum()
print(f"Probe pairs with rating found: {probe_with_ratings['rating'].notna().sum():,}")
print(f"Probe pairs missing from file 1 (are in files 2-4): {missing_probe:,}")

# ── Create train / test flags in the main DataFrame ──────────────────────────
# Create a set of (movie_id, user_id) tuples for O(1) lookup
probe_set = set(zip(probe_pairs["movie_id"], probe_pairs["user_id"]))

df["in_probe"] = df.apply(
    lambda r: (r["movie_id"], r["user_id"]) in probe_set, axis=1
)

df_train = df[~df["in_probe"]].copy()
df_test  = df[df["in_probe"]].copy()

print(f"\nTrain set size: {len(df_train):,}  ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test  set size: {len(df_test):,}  ({len(df_test)/len(df)*100:.1f}%)")

## 14. Save Cleaned Data to Parquet

We save the processed data in Parquet format because:
- It's 5–10× smaller than CSV for this type of data
- It preserves data types (int8 rating, datetime date) — no re-casting on load
- Much faster to read back in subsequent notebooks (Phase 2, 3, etc.)

In [ ]:
# Save full joined dataframe
df.to_parquet("ratings_with_movies.parquet", index=False)

# Save train / test splits separately for easy loading in Phase 2
df_train.drop(columns=["in_probe"]).to_parquet("train.parquet", index=False)
df_test.drop(columns=["in_probe"]).to_parquet("test.parquet",  index=False)

# Save movie metadata
df_movies.to_parquet("movies.parquet", index=False)

print("Files saved:")
for fname in ["ratings_with_movies.parquet", "train.parquet", "test.parquet", "movies.parquet"]:
    size_mb = os.path.getsize(fname) / 1e6
    print(f"  {fname:35s}  {size_mb:.1f} MB")

## 15. EDA Summary — Key Findings

Compile all observations into a single summary.
Copy these numbers directly into your report's EDA section.

In [ ]:
print("=" * 60)
print("  EDA SUMMARY — KEY FINDINGS")
print("=" * 60)

print("\n[Dataset size — file 1 only]")
print(f"  Ratings:          {n_ratings:>12,.0f}")
print(f"  Users:            {n_users:>12,.0f}")
print(f"  Movies:           {n_movies:>12,.0f}")
print(f"  Matrix density:   {n_ratings/(n_users*n_movies)*100:>11.4f}%")

print("\n[Rating distribution]")
for star in [1,2,3,4,5]:
    cnt = (df["rating"] == star).sum()
    print(f"  {star}-star: {cnt:>9,.0f}  ({cnt/n_ratings*100:.1f}%)")
print(f"  Mean:     {df['rating'].mean():.3f}")
print(f"  Std:      {df['rating'].std():.3f}")

print("\n[User activity]")
print(f"  Median ratings/user:   {ratings_per_user.median():.0f}")
print(f"  Mean   ratings/user:   {ratings_per_user.mean():.1f}")
print(f"  Max    ratings/user:   {ratings_per_user.max():,.0f}")
print(f"  Cold-start (<5):       {(ratings_per_user < 5).sum():,} users")

print("\n[Movie popularity]")
print(f"  Median ratings/movie:  {ratings_per_movie.median():.0f}")
print(f"  Mean   ratings/movie:  {ratings_per_movie.mean():.1f}")
print(f"  Max    ratings/movie:  {ratings_per_movie.max():,.0f}")
print(f"  Long-tail (<10 rtgs):  {(ratings_per_movie < 10).sum():,} movies")

print("\n[Train / Test split]")
print(f"  Train rows:    {len(df_train):>10,.0f}")
print(f"  Test  rows:    {len(df_test):>10,.0f}")
print(f"  Test fraction: {len(df_test)/len(df)*100:.2f}%")
print("=" * 60)

## Phase 1 Complete

### What you now have:
- `ratings_with_movies.parquet` — full cleaned DataFrame (movie_id, user_id, rating, date, title, year)
- `train.parquet` — training set (probe rows removed)
- `test.parquet`  — test set (probe pairs with known ratings)
- `movies.parquet` — movie metadata
- 7 EDA plots saved as PNG

### Key insights going into Phase 2:
1. **Positive rating bias** — mean rating ~3.6, 4-star is most common. Models predicting the mean have a natural edge.
2. **Heavy-tailed user activity** — most users rated very few movies; a small group dominates.
3. **Long-tail movie distribution** — top 100 movies have disproportionate coverage; niche movies are data-sparse.
4. **98%+ sparsity** — standard collaborative filtering needs matrix factorization to generalise.
5. **Temporal structure** — volume grows strongly 2004–2005; time-aware splits are preferable.

### Next: Phase 2 — Baseline Models (User-Based CF & Item-Based CF)
Load `train.parquet` and `test.parquet` and build your first collaborative filtering models.